# Bosonic Fractional Chern Insulator on the Haldane Honeycomb Lattice

**Abstract.** This notebook is a pedagogical, hands-on tour of the **bosonic fractional Chern
insulator (FCI)** realized by hard-core bosons on the Haldane honeycomb lattice. We derive the
Haldane tight-binding model (nearest-, next-nearest- and next-next-nearest-neighbor hoppings, with
a complex next-nearest-neighbor flux that breaks time-reversal symmetry), add the extended
Bose–Hubbard interactions, and impose the hard-core constraint. At half filling of the lower Chern
band ($\nu=1/2$, i.e. 3 bosons on the $2\times3$ torus and 6 bosons on $3\times4$) the exact
diagonalization yields **two nearly-degenerate ground states** — the finite-size fingerprint of the
bosonic Laughlin/semion state (topological ground-state degeneracy GSD$=2$ on the torus). We
reproduce the benchmark energies (agreement with the Julia reference to $\lesssim 2\times10^{-13}$),
and diagnose the topological order with the **spectrum flow** and the **fractional charge pump**
($\Delta Q\approx 1/2$).

**References.**
1. Y.-F. Wang, Z.-C. Gu, C.-D. Gong, D. N. Sheng, *Fractional quantum Hall effect of hard-core bosons in topological flat bands* (the "Sheng et al." bosonic FCI), Phys. Rev. Lett. **107**, 146803 (2011).
2. F. D. M. Haldane, *Model for a quantum Hall effect without Landau levels: Condensed-matter realization of the "parity anomaly"*, Phys. Rev. Lett. **61**, 2015 (1988).
3. D. N. Sheng, Z.-C. Gu, K. Sun, L. Sheng, *Fractional quantum Hall effect in the absence of Landau levels*, Nat. Commun. **2**, 389 (2011).
4. R. B. Laughlin, *Anomalous quantum Hall effect: An incompressible quantum fluid with fractionally charged excitations*, Phys. Rev. Lett. **50**, 1395 (1983).


## The Haldane Honeycomb Model

The honeycomb lattice has two sublattices ($A,B$) per unit cell, with primitive lattice vectors

\begin{equation}
\mathbf a_1 = (1,0),\qquad \mathbf a_2 = (1/2,\sqrt3/2),
\end{equation}

and basis $\boldsymbol\delta_A=(0,0)$, $\boldsymbol\delta_B=(1/3,1/3)$. Haldane's insight was that a
complex **next-nearest-neighbor** (NNN) hopping can break time-reversal symmetry and open a
topological gap *without* any net magnetic flux through the unit cell.

### Hopping terms (NN / NNN / NNNN)

Following Sheng *et al.*, the single-particle Hamiltonian has three hopping ranges:

- **Nearest-neighbor** (inter-sublattice, real): amplitude $-t$, with $t=1$.
- **Next-nearest-neighbor** (intra-sublattice, complex): amplitude $-t' e^{\pm i\phi}$ with
  $t'=0.60$, $\phi=0.4\pi$. The $\pm$ sign alternates with the chirality of the NNN hop, so a
  particle circling a plaquette acquires an Aharonov–Bohm phase $2\phi$ — the **Haldane flux**.
- **Next-next-nearest-neighbor** (inter-sublattice, real): amplitude $-t''$ with $t''=-0.58$; this
  third hopping is tuned to make the Chern band as flat as possible.

\begin{equation}
H_0 = -t\!\!\sum_{\langle i,j\rangle} b_i^\dagger b_j
      - t'\!\!\sum_{\langle\!\langle i,j\rangle\!\rangle} e^{\pm i\phi}\, b_i^\dagger b_j
      - t''\!\!\sum_{\langle\!\langle\!\langle i,j\rangle\!\rangle\!\rangle} b_i^\dagger b_j
      + \mathrm{h.c.}
\end{equation}

Each band has Chern number $C=\pm 1$; the lower band is nearly flat. Filling this flat Chern band
with interacting particles is the standard route to an FCI.


## The Extended Bose–Hubbard Model and the Hard-Core Constraint

Adding density–density interactions gives the **extended Bose–Hubbard model**:

\begin{equation}
H = H_0 + V_1\!\!\sum_{\langle i,j\rangle} n_i n_j + V_2\!\!\sum_{\langle\!\langle i,j\rangle\!\rangle} n_i n_j ,
\qquad n_i = b_i^\dagger b_i .
\end{equation}

For the FCI phase we set $V_1=V_2=0$: the topology alone, combined with the **hard-core
constraint** $(b_i^\dagger)^2=0$ (at most one boson per vertex), is enough to stabilize the
fractional state. The hard-core constraint is exactly what the bitmask encoding implements — each
vertex is a single bit $n_i\in\{0,1\}$.

The ED engine works on the **flattened graph**: the $2\times3$ torus has $2\cdot2\cdot3=12$
vertices and $\binom{12}{3}=220$ configurations at $\nu=1/2$ band filling (3 bosons); the
$3\times4$ torus has $24$ vertices and $\binom{24}{6}=134{,}596$ configurations (6 bosons).
Translation symmetry $\mathbb Z_{L_1}\times\mathbb Z_{L_2}$ reduces each momentum sector by a
factor $\sim|G|$.


## Exact Diagonalization on $[2,3]$: the Semion Doublet

We build the model, resolve the $\mathbb Z_2\times\mathbb Z_3$ translation symmetry, and run the
symmetry-resolved scan (matrix mode). The two nearly-degenerate ground states live in the momentum
sectors $k=(0,0)$ and $k=(1,0)$.


In [ ]:
import math
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def sector_vals(ed_data, k):
    for idx, irrep in enumerate(ed_data.irrep_list):
        if irrep.label == k:
            return ed_data.ed_scan_res[idx][0]   # 0-based ed_scan_res key
    raise KeyError(k)

sample_size = [2, 3]
model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_DNSheng)
lattice = model.lattice
n_filled = 3   # half band filling → 1/4 of the 12 flattened vertices

G = ed.build_translation_group(lattice)
ed_data = ed.build_ed_data(
    model, filling_fraction=Fraction(n_filled, lattice.n_site), symmetry_group=G)
print(f"Full Hilbert space dim: {math.comb(lattice.n_site, n_filled)}")
print(f"Orbits: {len(ed_data.orbit_catalog.representative_mask_list)}")
print(f"Irreps: {len(ed_data.irrep_list)}  (momenta (k1,k2))")

ed.ed_scan(ed_data, nev=5, mode="matrix")

e00 = float(sector_vals(ed_data, (0, 0))[0])
e10 = float(sector_vals(ed_data, (1, 0))[0])
print(f"\nE0(k=(0,0)) = {e00:.15f}")
print(f"E1(k=(1,0)) = {e10:.15f}")
print(f"topological splitting ΔE = {e10 - e00:.3e}")

# the two ground states are the semion FCI doublet (GSD = 2)
all_vals = sorted(float(v) for vals in ed_data.ed_scan_res.values() for v in vals)
print(f"lowest 6 eigenvalues: {[f'{x:.6f}' for x in all_vals[:6]]}")

fig, ax = ed.plot_spectrum(ed_data, shift_to_zero=True)
fig.savefig(os.path.join(FIG_DIR, "bosonic_FCI_spectrum_23.svg"))
print("saved spectrum → doc/figures/bosonic_FCI_spectrum_23.svg")


**Alignment with the Julia reference.** On this machine (ARPACK, `nev=5`), the Julia package
gives $E_0(k{=}(0,0)) = -7.163805363423441$ and $E_1(k{=}(1,0)) = -7.16337536157$; the Python port
reproduces both to better than $2\times10^{-13}$. The splitting $\Delta E\approx 4.3\times10^{-4}$
is the finite-size topological splitting of the $\nu=1/2$ bosonic Laughlin state on the torus —
matching the published DMRG benchmark.


## Exact Diagonalization on $[3,4]$: the Larger FCI

On the $3\times4$ torus (24 vertices, 6 bosons) the two FCI states live in the sectors $k=(0,0)$
and $k=(0,2)$. This is a full 12-sector scan (sector dimension $\approx 11{,}000$); it takes a few
minutes after the one-time JIT compilation.


In [ ]:
import math
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def sector_vals(ed_data, k):
    for idx, irrep in enumerate(ed_data.irrep_list):
        if irrep.label == k:
            return ed_data.ed_scan_res[idx][0]
    raise KeyError(k)

sample_size = [3, 4]
model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_DNSheng)
lattice = model.lattice
n_filled = 6   # half band filling → 1/4 of the 24 flattened vertices

G = ed.build_translation_group(lattice)
ed_data = ed.build_ed_data(
    model, filling_fraction=Fraction(n_filled, lattice.n_site), symmetry_group=G)
print(f"Full Hilbert space dim: {math.comb(lattice.n_site, n_filled)}")
print(f"Orbits: {len(ed_data.orbit_catalog.representative_mask_list)}")
print(f"Irreps: {len(ed_data.irrep_list)}")

ed.ed_scan(ed_data, nev=5, mode="matrix")   # ~2-3 min

for k in [(0, 0), (0, 2)]:
    vals = [float(v) for v in sector_vals(ed_data, k)[:5]]
    print(f"k={k}: {[f'{v:.12f}' for v in vals]}")

fig, ax = ed.plot_spectrum(ed_data, shift_to_zero=True)
fig.savefig(os.path.join(FIG_DIR, "bosonic_FCI_spectrum_34.svg"))
print("saved spectrum → doc/figures/bosonic_FCI_spectrum_34.svg")


**Verified alignment numbers** (Julia ARPACK, `nev=5`; Python agrees to $<2\times10^{-13}$):

| Sector | lowest five eigenvalues |
|---|---|
| $k=(0,0)$ | $-14.301456146831018$, $-13.722130823772558$, $-13.646661890699379$, $-13.640042308564075$, $-13.550170758301450$ |
| $k=(0,2)$ | $-14.299362590626847$, $-13.743011957063569$, $-13.717272063933615$, $-13.586585123115862$, $-13.554354573641792$ |

The two ground states $-14.301456\ldots$ and $-14.299362\ldots$ are nearly degenerate — the semion
doublet at $k=(0,0)$ and $k=(0,2)$.


## Spectrum Flow under Flux Insertion

Threading a flux $\theta$ through the $x$-periodic boundary and tracking the low-lying levels of
the FCI sectors is a direct diagnostic of the topological order: the two semion ground states
**cyclically exchange** under one flux quantum (they return only after two quanta, the
$T$-transformation of the modular group on the torus). We scan 7 flux points with `nev=3` to keep
the cell snappy (the package default is 9 points).


In [ ]:
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

sample_size = [2, 3]
model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_DNSheng)
labels = ed.default_fci_sectors(sample_size)          # [(0,0), (1,0)]
flux_list = list(np.linspace(0.0, 1.0, 7))           # 7 flux points (default 9)

flow = ed.flux_spectrum_flow(
    model, labels,
    filling_fraction=Fraction(1, 4),
    flux_direction=1,
    twisted_phases_over_2π_list=flux_list,
    nev=3,
    fig_path=os.path.join(FIG_DIR, "bosonic_FCI_spectrum_flow_23.svg"),
    checkpoint_dir=CKPT_DIR,
)
print("ground-state energy per flux point, per sector:")
print(flow.energies[:, :, 0])


## Fractional Charge Pump: $\Delta Q \approx 1/2$

The **Laughlin charge pump** threads one flux quantum through the cylinder and tracks the winding of
Resta's periodic position operator $U_y=\exp\big(\tfrac{2\pi i}{L_y}\sum_j x_{j,y}\hat n_j\big)$ in
the transverse direction. Each of the two semion polarization branches winds by **half a charge**
per flux quantum ($\Delta Q\approx 0.5$), summing to $1$ — the many-body Chern number. The
`flux_charge_pump` call reuses the per-flux checkpoints produced by the spectrum-flow cell above.


In [ ]:
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

sample_size = [2, 3]
model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_DNSheng)
labels = ed.default_fci_sectors(sample_size)          # [(0,0), (1,0)]
flux_list = list(np.linspace(0.0, 1.0, 7))

pump = ed.flux_charge_pump(
    model, labels,
    filling_fraction=Fraction(1, 4),
    flux_direction=1,              # thread θ along x
    polarization_direction=2,      # measure U_y (transverse)
    twisted_phases_over_2π_list=flux_list,
    nev_per_sector=1,              # one ground state per sector
    fig_path=os.path.join(FIG_DIR, "bosonic_FCI_charge_pump_23.svg"),
    checkpoint_dir=CKPT_DIR,
)
print("pumped charges ΔQ =", pump.pumped_charges)
print("|ΔQ| per branch ≈ 0.5 → total =", round(abs(pump.pumped_charges).sum(), 6))


The result $|\Delta Q|\approx(0.5,0.5)$ is the **semion topological order**: each branch
carries a fractional charge $e/2$ per flux quantum, and the two branches together pump exactly one
charge. This is the finite-size analogue of the $\nu=1/2$ bosonic Laughlin state.

---

*This notebook is part of `realspace_exactdiagonalization_py`. The model builder lives in
`models/bosonic_fci.py`; the spectrum-flow and charge-pump observables live in
`observables/spectrum_flow.py` and `observables/charge_pump.py`.*
